In [2]:
from parse import parse
from typing import List, Dict
from pathlib import Path
import pandas as pd
import now
import datetime
import os
import gzip
import shutil
from typing import Union

def collect_matching_lines(log_text: str, identifiers: Dict[str, List[str]]) -> Dict[str, List[str]]:

    # Parse pattern for the log format
    log_pattern = "{timestamp} (PID: {pid}) (TID: {tid}) {rest}" # rest matches everything after.

    results = {identifier: [] for identifier in identifiers}

    for line in log_text.strip().split('\n'):
        line = line.strip()

        # Try to parse the line
        parsed = parse(log_pattern, line)

        if parsed:  # Line matches the log format
            timestamp = parsed['timestamp']
            pid = parsed['pid']
            tid = parsed['tid']

            # Check each identifier
            for identifier_name, required_strings in identifiers.items():
                # Check if ALL required strings are present in the line
                if all(s in line for s in required_strings):
                    results[identifier_name].append((timestamp, pid, tid, line))

    return results


# Example usage
if __name__ == "__main__":

    now = datetime.datetime.now()
    timestamp_str = now.strftime("%Y-%m-%d_%H-%M-%S")
    # --- 2. Construct the New File Name ---
    base_dir = "C:/Users/MohitKumar/Downloads/MohitAgentLogs_base/MohitAgentLogs/analysis_results"
    base_name = "stats_output"
    file_extension = ".txt"
   
    identifiers = {
        "VMAgentAPIUpdateInformation_start": ["vmagentapiupdateinformationrequest.cpp", "VMAgentAPIUpdateInformationRequest::formRequest"],
        "VMAgentAPIUpdateInformation_end": ["vmagentmodule.cpp", "VMAgentModule::onHandleUpdateInfoResponse"],

        "AgentAPIUpdateSessions_start": ["vmworkflowcontrol.cpp:365", "Got send update VM sessions event"],
        "AgentAPIUpdateSessions_end": ["vmagentmodule.cpp:466", "Session update sent as next timer reset"],

        "AgentAPIUpdateResourceUsage_start": ["TopResourceUsageProcesses::collectSample"],
        "AgentAPIUpdateResourceUsage_end": ["perfmonitor.cpp", "PerfMonitor Resource samples saved"],

        "AgentAPIDataCollection_start": ["o1workflowcontrol.cpp:895","Got send data collection event for"],
        "AgentAPIDataCollection_end": ["o1datacollection.cpp:30", "~O1DataCollection()"]
    }
    
    # New filename format: stats_output_2025-12-05_14-52-41.txt
    new_file_name = f"{base_name}_{timestamp_str}{file_extension}"

    # Combine the directory and the new filename
    output_log_file_name = os.path.join(base_dir, new_file_name)

    #print(output_log_file_name)
    Logs_dir="C:/Users/MohitKumar/Downloads/MohitAgentLogs_base/MohitAgentLogs/logs_actual"
    log_folder_path = Path(Logs_dir)
    
    #----------------------------------
    target_dir = Path(Logs_dir)
    
    print(f"--- Starting decompression in: {target_dir} ---")
    for gzip_file_path in target_dir.glob("*.gzip"):
        
        #zip_file_name = gzip_file_path.stem 
        if gzip_file_path.suffixes and gzip_file_path.suffixes[-1] == '.gzip':
            # Create the final path by removing the last suffix (.gz)
            output_file_path = gzip_file_path.with_suffix('')
        else:
             # Should not happen if glob is correct, but handles single extension better
            output_file_path = target_dir / (gzip_file_path.name.replace('.gzip', ''))

        #print(f"  - Decompressing {gzip_file_path.name} to {output_file_path.name}")
        
        try:
            # Open the gzipped file in binary read mode ('rb')
            with gzip.open(gzip_file_path, 'rb') as f_in:
                # Open the output file in binary write mode ('wb')
                with open(output_file_path, 'wb') as f_out:
                    # Copy the contents from the compressed file to the new file
                    shutil.copyfileobj(f_in, f_out)
            
        except OSError as e:
            # Handles errors like file permission issues or corrupted gzip files
            print(f"  - ERROR: Could not process {gzip_file_path.name}. Reason: {e}")
        except Exception as e:
            print(f"  - AN UNEXPECTED ERROR occurred with {gzip_file_path.name}: {e}")
    print("--- Decompression complete ---")
    #-------------------------------------------------
    if not log_folder_path.is_dir():
        print(f"Error: The directory '{log_folder_path}' was not found.")
    else:
        print(f"--- Processing files in folder: {log_folder_path} ---")
        
        # **THIS BLOCK OPENS THE FILE**
        # Everything that uses 'f' must be inside here.
        with open(output_log_file_name, 'a') as f: 
            #f.write(f"\n\n{'#' * 60}\n")
            #f.write(f"INDIVIDUAL FILE STATISTICS START ({timestamp_str})\n")
            #f.write(f"{'#' * 60}\n\n")
    
            # Loop 1: Iterates over all .log files (This contains the data collection)
            for file_path in log_folder_path.glob("*.log"):
                if file_path.is_file():
                    print(f"\n--- Reading file: {file_path.name} ---")
                    sample_log="none"
                    try:
                        sample_log = file_path.read_text(encoding='utf-8')
                        results = collect_matching_lines(sample_log, identifiers)
                        timestamp_list = []
                        for identifier, matches in results.items():
                            #print(f"\n{identifier}:")
                            #print(len(matches))
                            for timestamp, pid, tid, line in matches:               
                                timestamp_list.append([identifier, timestamp, pid, tid])
                        
                        df = pd.DataFrame(timestamp_list, columns=['identifier', 'timestamp', 'pid', 'tid'])
                        # Strip whitespace, replace colon with dot, then parse
                        df['timestamp'] = pd.to_datetime(df['timestamp'].str.strip().str.replace(r':(\d{3})$', r'.\1', regex=True), format='%Y-%m-%dT%H:%M:%S.%f')
                        #print(df['timestamp'])
                        df = df.sort_values('timestamp')
                                        
                        pd.set_option('display.float_format', '{:.2f}'.format)
                        df["identifier_prefix"] = df['identifier'].str.split("_").str[0]
                        unique_list = df["identifier_prefix"].unique().tolist()
                                            
                        with open(output_log_file_name, 'a') as f:
                            for identifier_prefix in unique_list:
                                df_prefix = df[df['identifier_prefix'] == identifier_prefix].copy()
                                df_prefix_dedup = df_prefix.groupby((df_prefix['identifier'] != df_prefix['identifier'].shift()).cumsum()).last().reset_index(drop=True)
                                df_prefix_start = df_prefix_dedup[df_prefix_dedup['identifier'].str.contains('_start')].reset_index(drop=True).copy()
                                df_prefix_end = df_prefix_dedup[df_prefix_dedup['identifier'].str.contains('_end')].reset_index(drop=True).copy()
                                df_merged = pd.concat([df_prefix_start.add_suffix('_start'), df_prefix_end.add_suffix('_end')], axis=1)
    
                                #if (df_merged['timestamp_end'] - df_merged['timestamp_start']).dt.total_seconds() * 1000 > 0:
                                df_merged['time_diff_ms'] = (df_merged['timestamp_end'] - df_merged['timestamp_start']).dt.total_seconds() * 1000
    
                                #0-------df_merged.dropna(subset=['time_diff_ms'], inplace=True)
                                #---------df_merged = df_merged[df_merged['time_diff_ms'] > 0].copy()
                    
                                stats = df_merged['time_diff_ms'].describe()
                                iteration_count = stats['count']
                                min_time = stats['min']
                                max_time = stats['max']
                                #print(f"Total Count of Iterations/Successful Completions: {iteration_count:.0f}")
                                
                                quantiles = df_merged['time_diff_ms'].quantile([0.25,0.50,0.75,0.90,0.95,0.99])
                                quantiles.index = ['25%','50%','75%','90%', '95%', '99%']
                                """
                                #f.write(f"{'==' * 20}\n")
                                f.write(f"identifier_prefix: {identifier_prefix}\n")
                                f.write(f"Total Count of Iterations/Successful Completions: {iteration_count:.0f}\n")
                                f.write(f"Min: {stats['min']:.2f}\n")
                                f.write(f"{quantiles.to_string()}\n")
                                f.write(f"Max: {stats['max']:.2f}\n")
                                #f.write(f"{'==' * 20}\n")
                                """
                                                            
                                #f.write(f"here is the stats ")
                                #f.write(f"{df_merged['time_diff_ms'].describe()}\n")
                                #f.write(f"here is the stats complete")
                                
                        #print(sample_log[:500] + "...") 
            
                    except UnicodeDecodeError:
                        print(f"Error: Could not decode '{file_path.name}'. Skipping.")
                    except Exception as e:
                        print(f"An unexpected error occurred while reading '{file_path.name}': {e}. Skipping.")
                
                elif file_path.is_dir():
                    print(f"--- Skipping directory: {file_path.name} ---")
                
                # ... (End of file processing loop) ...
    
            # ------------------------------------------------------------------
            # **START OF THE OVERALL SUMMARY BLOCK (MUST BE INSIDE 'with f:')**
            # ------------------------------------------------------------------
            
            # Combine all DataFrames into one
    all_merged_dfs[]
    if all_merged_dfs:
        final_df = pd.concat(all_merged_dfs, ignore_index=True)

        # 2. Group the resulting final_df to get all prefixes' stats
        overall_stats = final_df.groupby('identifier_prefix')['time_diff_ms'].describe()
        """
        df["identifier_prefix"] = df['identifier'].str.split("_").str[0]
        unique_list = df["identifier_prefix"].unique().tolist()
        for identifier_prefix in unique_list:
            #if 'time_diff_ms' in df_merged.columns and not df_merged['time_diff_ms'].empty:                   
            df_merged['identifier_prefix'] = identifier_prefix 
            # Ensure you are appending the data for the current prefix
            all_merged_dfs.append(df_merged[['identifier_prefix', 'time_diff_ms']].copy())
        """
        final_df = pd.concat(all_merged_dfs, ignore_index=True)
        print(f"Prefixes found in final_df: {final_df['identifier_prefix'].unique()}")
        overall_stats = final_df.groupby('identifier_prefix')['time_diff_ms'].describe()
        with open(output_log_file_name, 'a') as f_summary:    
            f_summary.write(f"\n\n{'#' * 60}\n")
            f_summary.write(f"OVERALL SUMMARY STATISTICS ACROSS ALL FILES\n")
            f_summary.write(f"{'#' * 60}\n")

            # Group the combined data by prefix and calculate descriptive statistics
            for prefix in overall_stats.index:
                stats = overall_stats.loc[prefix]
                
                # Recalculate quantiles from the full data (describe() only gives 25%, 50%, 75%)
                full_prefix_data = final_df[final_df['identifier_prefix'] == prefix]['time_diff_ms']
                quantiles = full_prefix_data.quantile([0.25,0.50,0.75,0.90,0.95,0.99])
                quantiles.index = ['25%','50%','75%','90%', '95%', '99%']
                
                f_summary.write(f"\n\n{'++' * 20}\n")
                f_summary.write(f"OVERALL STATS FOR: {prefix}\n")
                f_summary.write(f"Total Count of Cycles (All Files): {stats['count']:.0f}\n")
                f_summary.write(f"Mean Time: {stats['mean']:.2f} ms\n")
                f_summary.write(f"Std Dev: {stats['std']:.2f} ms\n")
                f_summary.write(f"Min: {stats['min']:.2f} ms\n")
                f_summary.write(f"{quantiles.to_string()}\n")
                f_summary.write(f"Max: {stats['max']:.2f} ms\n")
                f_summary.write(f"{'++' * 20}\n")

    else:
        # THIS LINE (THE ONE CAUSING THE ERROR) IS NOW INSIDE THE 'with open' BLOCK
        f.write("\n\nNo valid log cycles were found across any file to generate a final summary.\n")
                    
    # **END OF THE 'with open' BLOCK** - The file is now closed.
    
    #print(f"\n--- All processing complete. Results written to: {output_log_file_name} ---

SyntaxError: invalid syntax (2424685139.py, line 194)